# DBRepo View Loading

This notebook creates all seven views for the Unemployment Prediction ML Pipeline using direct HTTP calls to the DBRepo REST API.

Views created:
- `ml_feature_table` -> base denormalised feature table
- `train_split` -> years 2002-2015
- `validation_split` -> years 2016-2018
- `test_split` -> years 2019+ (excl. 2020/2021)
- `inner_city_districts` -> inner-city district data
- `outer_city_districts` -> outer-city district data
- `gender_disaggregated_features` -> Male/Female breakdown

**Prerequisites:** the four tables (`district`, `measurement_info`, `tourism`, `unemployment`) must already exist in the target database.

## 0 · Imports & configuration

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
load_dotenv(os.path.join(os.path.dirname(__file__), '..', 'config', '.env'))

ENDPOINT    = "https://test.dbrepo.tuwien.ac.at"
DATABASE_ID = "412fb0ce-5299-4d0e-a271-4641b1365b8a"
USERNAME    = os.getenv("DBREPO_USERNAME")
PASSWORD    = os.getenv("DBREPO_PASSWORD")

AUTH    = (USERNAME, PASSWORD)
HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
BASE    = f"{ENDPOINT}/api/v1/database/{DATABASE_ID}"

r = requests.get(f"{ENDPOINT}/api/v1/user/{USERNAME}", auth=AUTH, headers=HEADERS)
if not r.ok:
    raise RuntimeError(f"Auth failed ({r.status_code}): {r.text[:200]}")
print(f"Connected as: {USERNAME}  (HTTP {r.status_code})")

Connected as: Binw3g  (HTTP 200)


## 1 · Resolve column & operator UUIDs

In [ ]:
# Fetch all tables, then fetch each individually for columns
r = requests.get(f"{BASE}/table", auth=AUTH, headers=HEADERS)
r.raise_for_status()
tables_brief = r.json()

REQUIRED = {"district", "measurement_info", "tourism", "unemployment"}
found    = {t["name"] for t in tables_brief}
missing  = REQUIRED - found
if missing:
    raise RuntimeError(f"Missing tables: {missing} - run the data-loading notebook first.")

# Table UUID map: name -> table_id
TID = {t["name"]: t["id"] for t in tables_brief}

# Fetch each table individually to get the full schema (including columns)
CID = {}
for tname, tid in TID.items():
    r = requests.get(f"{BASE}/table/{tid}", auth=AUTH, headers=HEADERS)
    r.raise_for_status()
    full_table = r.json()
    for col in full_table.get("columns", []):
        CID[(tname, col["name"])] = col["id"]

print("Table UUIDs:")
for name, tid in sorted(TID.items()):
    print(f"  {name:<20} {tid}")
print()
print("Column UUIDs:")
for (tname, cname), cid in sorted(CID.items()):
    print(f"  {tname:<20} {cname:<20} {cid}")

Table UUIDs:
  district             886fba43-0570-4dda-8233-4f73473ca0a4
  measurement_info     5d6d28a7-f211-4bc4-9ada-94450cd0fe6f
  tourism              fa5fe819-9b20-422a-a6fe-299ba0043d87
  unemployment         5a3f74b9-8a62-4f33-9486-6c6cd3d34cd2

Column UUIDs:
  district             district_code        603fea47-8ff3-4ec4-a426-a4b891b16b8a
  district             district_id          2998a6be-d47d-447e-b20c-4b42f232e6b6
  district             nuts_code            8c391fab-1ae3-4a8c-bc04-6fe83b47e823
  measurement_info     district_id          e98390d6-07a3-4f8c-b112-33fcf9f77649
  measurement_info     measurement_id       00d8526b-c469-47ea-b8fe-06bf4f981ffb
  measurement_info     population_avg       e78e310d-1409-4b62-b078-e677e2adac1b
  measurement_info     reference_date       ad45fb3b-a537-41c0-8194-8a852aa85f18
  tourism              density              dd855875-fc15-42ea-9536-f88368054759
  tourism              measurement_id       dc37b7d0-3257-4d90-a6fd-dfcbdcd767a7
  t

In [3]:
# Fetch operator UUIDs 
r = requests.get(f"{BASE}", auth=AUTH, headers=HEADERS)
r.raise_for_status()
image_id = r.json()["container"]["image"]["id"]

r = requests.get(f"{ENDPOINT}/api/v1/image/{image_id}", auth=AUTH, headers=HEADERS)
r.raise_for_status()
OID = {op["value"]: op["id"] for op in r.json().get("operators", [])}

print("Operator UUIDs:")
for sym, oid in sorted(OID.items()):
    print(f"  {sym:<6} {oid}")

Operator UUIDs:
  !=     130c6cd2-f830-11f0-8413-0e74369233d4
  <      130c6a49-f830-11f0-8413-0e74369233d4
  <=     130c6ae1-f830-11f0-8413-0e74369233d4
  <=>    130c693b-f830-11f0-8413-0e74369233d4
  =      130c6782-f830-11f0-8413-0e74369233d4
  >      130c6bdd-f830-11f0-8413-0e74369233d4
  >=     130c6c65-f830-11f0-8413-0e74369233d4
  IN     130c6e35-f830-11f0-8413-0e74369233d4
  IS NOT NULL 130c6f12-f830-11f0-8413-0e74369233d4
  IS NULL 130c6f8a-f830-11f0-8413-0e74369233d4
  LIKE   130c6d46-f830-11f0-8413-0e74369233d4
  NOT IN 130c6ea3-f830-11f0-8413-0e74369233d4
  NOT LIKE 130c6db7-f830-11f0-8413-0e74369233d4
  NOT REGEXP 130c705c-f830-11f0-8413-0e74369233d4
  REGEXP 130c6ff0-f830-11f0-8413-0e74369233d4


## 2 · Helper functions

In [4]:
MI = "measurement_info"
D  = "district"
U  = "unemployment"
T  = "tourism"

def c(table: str, col: str) -> str:
    """Return the UUID for (table, column). Raises KeyError with context on miss."""
    key = (table, col)
    if key not in CID:
        known = [k[1] for k in CID if k[0] == table]
        raise KeyError(f"Column not found: {key}. Known columns for '{table}': {known}")
    return CID[key]

def op(symbol: str) -> str:
    """Return the UUID for an operator symbol (e.g. '=', '!=', '<=', '>=')."""
    if symbol not in OID:
        raise KeyError(f"Operator '{symbol}' not found. Available: {list(OID.keys())}")
    return OID[symbol]


def create_or_skip(name: str, query: dict) -> None:
    """POST a view to the REST API. HTTP 409 = already exists (skip)."""
    payload = {
        "name": name,
        "query": query,
        "is_public": True,
        "is_schema_public": True,
    }

    r = requests.post(f"{BASE}/view", auth=AUTH, headers=HEADERS, json=payload)
    if r.status_code == 201:
        print(f"  ✓ '{name}' created  (id={r.json().get('id', '?')})")
    elif r.status_code == 409:
        print(f"  ⚠ '{name}' already exists, skipping.")
    else:
        raise RuntimeError(
            f"Failed to create '{name}': HTTP {r.status_code}\n{r.text[:500]}"
        )


# Shared building blocks
def base_datasource_ids() -> list:
    return [TID[MI]]


def base_columns() -> list:
    """Seven output columns shared by ml_feature_table and all split views."""
    return [
        {"id": c(D, "district_code")},
        {"id": c(MI, "reference_date")},
        {"id": c(MI, "population_avg")},
        {"id": c(U, "value"), "alias": "uep_value"},
        {"id": c(U, "density"), "alias": "uep_density"},
        {"id": c(T, "value"), "alias": "tou_value"},
        {"id": c(T, "density"), "alias": "tou_density"},
    ]


def base_joins() -> list:
    """Three INNER JOINs: measurement_info -> district, unemployment, tourism."""
    return [
        {
            "type": "inner",
            "datasource_id": TID[D],
            "conditionals": [{"column_id": c(MI, "district_id"), "foreign_column_id": c(D, "district_id")}],
        },
        {
            "type": "inner",
            "datasource_id": TID[U],
            "conditionals": [{"column_id": c(MI, "measurement_id"), "foreign_column_id": c(U, "measurement_id")}],
        },
        {
            "type": "inner",
            "datasource_id": TID[T],
            "conditionals": [{"column_id": c(MI, "measurement_id"), "foreign_column_id": c(T, "measurement_id")}],
        },
    ]


def base_filters(extra: list | None = None) -> list:
    """gender='Both', exclude district 90000, exclude COVID years 2020/2021."""
    return [
        {"type": "where", "column_id": c(U,  "gender"), "operator_id": op("="),  "value": "Both"},
        {"type": "and"},
        {"type": "where", "column_id": c(D,  "district_code"), "operator_id": op("!="), "value": "90000"},
        {"type": "and"},
        {"type": "where", "column_id": c(MI, "reference_date"), "operator_id": op("!="), "value": "2020-12-31"},
        {"type": "and"},
        {"type": "where", "column_id": c(MI, "reference_date"), "operator_id": op("!="), "value": "2021-12-31"},
    ] + (extra or [])


def base_orders() -> list:
    return [
        {"column_id": c(D, "district_code"), "direction": "asc"},
        {"column_id": c(MI, "reference_date"), "direction": "asc"},
    ]

## 3 · Create views

### 3.1 · `ml_feature_table`

In [5]:
create_or_skip(
    name="ml_feature_table",
    query={
        "datasource_ids": base_datasource_ids(),
        "columns": base_columns(),
        "joins": base_joins(),
        "filters": base_filters(),
        "orders": base_orders(),
    },
)

  ✓ 'ml_feature_table' created  (id=2a10edf4-6645-4f6f-9bb5-83919cff02ea)


### 3.2 · `train_split`
Years 2002-2015 (~322 rows: 23 districts × 14 years).

In [6]:
create_or_skip(
    name="train_split",
    query={
        "datasource_ids": base_datasource_ids(),
        "columns": base_columns(),
        "joins": base_joins(),
        "filters": base_filters([
            {"type": "and"},
            {"type": "where", "column_id": c(MI, "reference_date"), "operator_id": op("<="), "value": "2015-12-31"},
        ]),
        "orders": base_orders(),
    },
)

  ✓ 'train_split' created  (id=3edb9ad6-78c1-48b7-afc6-f6c88b4a1a61)


### 3.3 · `validation_split`
Years 2016-2018 (~69 rows: 23 districts × 3 years).

In [7]:
create_or_skip(
    name="validation_split",
    query={
        "datasource_ids": base_datasource_ids(),
        "columns": base_columns(),
        "joins": base_joins(),
        "filters": base_filters([
            {"type": "and"},
            {"type": "where", "column_id": c(MI, "reference_date"), "operator_id": op(">="), "value": "2016-01-01"},
            {"type": "and"},
            {"type": "where", "column_id": c(MI, "reference_date"), "operator_id": op("<="), "value": "2018-12-31"},
        ]),
        "orders": base_orders(),
    },
)

  ✓ 'validation_split' created  (id=abba7dcd-e5ab-4f5c-9cb8-b1fc1bb7d6b5)


### 3.4 · `test_split`
Years 2019+ excl. 2020/2021 (~69 rows: 23 districts × 3 years).

In [8]:
create_or_skip(
    name="test_split",
    query={
        "datasource_ids": base_datasource_ids(),
        "columns": base_columns(),
        "joins": base_joins(),
        "filters": base_filters([
            {"type": "and"},
            {"type": "where", "column_id": c(MI, "reference_date"), "operator_id": op(">="), "value": "2019-01-01"},
        ]),
        "orders": base_orders(),
    },
)

  ✓ 'test_split' created  (id=9469de00-ad14-489b-a8f1-73e1ec21218c)


### 3.5 · `inner_city_districts`
High tourism inner-city districts (90100-90900) for EDA.

In [9]:
create_or_skip(
    name="inner_city_districts",
    query={
        "datasource_ids": base_datasource_ids(),
        "columns": base_columns(),
        "joins": base_joins(),
        "filters": base_filters([
            {"type": "and"},
            {"type": "where", "column_id": c(D,  "district_code"), "operator_id": op("<="), "value": "90900"},
        ]),
        "orders": base_orders(),
    },
)

  ✓ 'inner_city_districts' created  (id=5a624d3a-876e-4078-83b4-c0bbec256179)


### 3.6 · `outer_city_districts`
Outer-city districts (91000-92300) for EDA.

In [10]:
create_or_skip(
    name="outer_city_districts",
    query={
        "datasource_ids": base_datasource_ids(),
        "columns": base_columns(),
        "joins": base_joins(),
        "filters": base_filters([
            {"type": "and"},
            {"type": "where", "column_id": c(D,  "district_code"), "operator_id": op(">="), "value": "91000"},
        ]),
        "orders": base_orders(),
    },
)

  ✓ 'outer_city_districts' created  (id=96116d38-a9c9-4dda-b608-6fe65ade65ce)


### 3.7 · `gender_disaggregated_features`
Male/Female rows only, excl. district 90000 and COVID years.

In [11]:
create_or_skip(
    name="gender_disaggregated_features",
    query={
        "datasource_ids": base_datasource_ids(),
        "columns": base_columns() + [{"id": c(U, "gender")}],
        "joins": base_joins(),
        "filters": [
            {"type": "where", "column_id": c(U,  "gender"), "operator_id": op("!="), "value": "Both"},
            {"type": "and"},
            {"type": "where", "column_id": c(D,  "district_code"), "operator_id": op("!="), "value": "90000"},
            {"type": "and"},
            {"type": "where", "column_id": c(MI, "reference_date"), "operator_id": op("!="), "value": "2020-12-31"},
            {"type": "and"},
            {"type": "where", "column_id": c(MI, "reference_date"), "operator_id": op("!="), "value": "2021-12-31"},
        ],
        "orders": base_orders() + [{"column_id": c(U,  "gender"), "direction": "asc"}],
    },
)

  ✓ 'gender_disaggregated_features' created  (id=fe047ba9-61e1-433c-9bd7-52551ca15b33)


## 4 · Verify

In [12]:
EXPECTED = {
    "ml_feature_table", "train_split", "validation_split", "test_split", "inner_city_districts", "outer_city_districts", "gender_disaggregated_features",
}

r = requests.get(f"{BASE}/view", auth=AUTH, headers=HEADERS)
r.raise_for_status()
views = r.json()
registered = {v["name"] for v in views}
missing = EXPECTED - registered

if missing:
    print(f"✗ Not registered: {missing}")
else:
    print("✓ All seven views registered.")
    print()
    for v in sorted(views, key=lambda x: x["name"]):
        if v["name"] in EXPECTED:
            print(f"  {v['name']:<35} id={v['id']}")

✓ All seven views registered.

  gender_disaggregated_features       id=fe047ba9-61e1-433c-9bd7-52551ca15b33
  inner_city_districts                id=5a624d3a-876e-4078-83b4-c0bbec256179
  ml_feature_table                    id=2a10edf4-6645-4f6f-9bb5-83919cff02ea
  outer_city_districts                id=96116d38-a9c9-4dda-b608-6fe65ade65ce
  test_split                          id=9469de00-ad14-489b-a8f1-73e1ec21218c
  train_split                         id=3edb9ad6-78c1-48b7-afc6-f6c88b4a1a61
  validation_split                    id=abba7dcd-e5ab-4f5c-9cb8-b1fc1bb7d6b5


## 5 · Create PID identifiers for each view

Each view gets a citable persistent identifier via `POST /api/v1/identifier`. We resolve the view IDs from the registered views, then post one identifier per view.

In [ ]:
# Per-view title and description for the identifier payload
VIEW_METADATA = {
    "ml_feature_table": {
        "title": "ML Feature Table - Unemployment and Tourism Vienna Districts",
        "description": (
            "Main denormalised feature table for the unemployment prediction ML pipeline. "
            "Joins measurement_info, district, unemployment (gender='Both') and tourism across all 23 Vienna districts. "
            "Excludes district 90000 (Vienna-wide aggregate) and COVID outlier years 2020 and 2021. "
            "Covers years 2002-2023 (excl. 2020-2021). Base view for all train/val/test split views."
        ),
    },
    "train_split": {
        "title": "Training Split - Unemployment and Tourism Vienna Districts (2002-2015)",
        "description": (
            "Training portion of the chronological data split derived from ml_feature_table. "
            "Covers years 2002-2015 (322 rows: 23 districts x 14 years). "
            "Used to fit Linear Regression and Random Forest models for unemployment prediction."
        ),
    },
    "validation_split": {
        "title": "Validation Split - Unemployment and Tourism Vienna Districts (2016-2018)",
        "description": (
            "Validation portion of the chronological data split derived from ml_feature_table. "
            "Covers years 2016-2018 (69 rows: 23 districts x 3 years). "
            "Used for hyperparameter tuning during model development."
        ),
    },
    "test_split": {
        "title": "Test Split - Unemployment and Tourism Vienna Districts (2019, 2022, 2023)",
        "description": (
            "Test portion of the chronological data split derived from ml_feature_table. "
            "Covers years 2019, 2022, 2023 (69 rows: 23 districts x 3 years; COVID years 2020-2021 excluded). "
            "Held out entirely until final model evaluation."
        ),
    },
    "inner_city_districts": {
        "title": "Inner City Districts - Unemployment and Tourism Vienna (Districts 90100-90900)",
        "description": (
            "Feature table restricted to the nine inner-city Vienna districts (90100-90900) "
            "which exhibit disproportionately high tourism activity relative to their resident population. "
            "Derived from ml_feature_table. Useful for district-segment-specific models and EDA "
            "comparing inner-city vs. outer-district unemployment dynamics."
        ),
    },
    "outer_city_districts": {
        "title": "Outer City Districts - Unemployment and Tourism Vienna (Districts 91000-92300)",
        "description": (
            "Feature table restricted to the fourteen outer residential Vienna districts (91000-92300) "
            "with lower tourism density and more stable unemployment patterns. "
            "Derived from ml_feature_table. Complement of inner_city_districts; "
            "together they cover all 23 Vienna districts (excl. aggregate 90000)."
        ),
    },
    "gender_disaggregated_features": {
        "title": "Gender-Disaggregated Features - Unemployment and Tourism Vienna Districts",
        "description": (
            "Gender-disaggregated feature table including Male and Female unemployment rows "
            "(gender='Both' excluded). Includes the GENDER column for experiments that incorporate "
            "sex as an additional predictor variable. "
            "Excludes COVID years 2020-2021 and Vienna-wide district 90000."
        ),
    },
}

CREATORS = [
    {"creator_name": "Florian Angerer", "affiliation": "TU Wien", "name_type": "Personal"},
    {"creator_name": "Swetha Maria Siby", "affiliation": "TU Wien", "name_type": "Personal"},
    {"creator_name": "Nicolas Philipp", "affiliation": "TU Wien", "name_type": "Personal"},
    {"creator_name": "Midhun Suresh Nair", "affiliation": "TU Wien", "name_type": "Personal"},
]

def create_view_identifier(view_id: str, view_name: str) -> None:
    """Create a PID identifier for a view. Skips if one already exists."""
    meta = VIEW_METADATA[view_name]
    payload = {
        "database_id": DATABASE_ID,
        "type": "view",
        "view_id": view_id,
        "publication_year": 2026,
        "publisher": "Stadt Wien - Wirtschaft und Finanzen",
        "language": "en",
        "titles": [
            {"title": meta["title"], "language": "en"}
        ],
        "descriptions": [
            {"description": meta["description"], "language": "en", "type": "Abstract"}
        ],
        "funders": [],
        "licenses": [
            {
                "identifier": "CC-BY-4.0",
                "uri": "https://www.data.gv.at/info/netiquette?locale=de",
                "description": "Creative Commons Attribution 4.0 International",
            }
        ],
        "creators": CREATORS,
        "related_identifiers": [],
    }

    r = requests.post(
        f"{ENDPOINT}/api/v1/identifier",
        auth=AUTH, headers=HEADERS, json=payload
    )
    if r.status_code in (200, 201):
        data = r.json()
        print(f"  ✓ '{view_name}' identifier created  (doi={data.get('doi', data.get('id', '?'))})")
    elif r.status_code == 409:
        print(f"  ⚠ '{view_name}' identifier already exists - skipping.")
    else:
        raise RuntimeError(
            f"Failed to create identifier for '{view_name}': HTTP {r.status_code}\n{r.text[:400]}"
        )


# Resolve view name -> view ID from the already-registered views
r = requests.get(f"{BASE}/view", auth=AUTH, headers=HEADERS)
r.raise_for_status()
view_id_map = {v["name"]: v["id"] for v in r.json()}

print("=== Creating view identifiers ===")
for view_name in VIEW_METADATA:
    if view_name not in view_id_map:
        print(f"  ✗ '{view_name}' not found in registered views - skipping.")
        continue

    create_view_identifier(view_id_map[view_name], view_name)

print("\n✓ All view identifiers processed.")

=== Creating view identifiers ===
{'database_id': '412fb0ce-5299-4d0e-a271-4641b1365b8a', 'type': 'view', 'view_id': '2a10edf4-6645-4f6f-9bb5-83919cff02ea', 'publication_year': 2026, 'publisher': 'Stadt Wien - Wirtschaft und Finanzen', 'language': 'en', 'titles': [{'title': 'ML Feature Table - Unemployment and Tourism Vienna Districts', 'language': 'en'}], 'descriptions': [{'description': "Main denormalised feature table for the unemployment prediction ML pipeline. Joins measurement_info, district, unemployment (gender='Both') and tourism across all 23 Vienna districts. Excludes district 90000 (Vienna-wide aggregate) and COVID outlier years 2020 and 2021. Covers years 2002-2023 (excl. 2020-2021). Base view for all train/val/test split views.", 'language': 'en', 'type': 'Abstract'}], 'funders': [], 'licenses': [{'identifier': 'CC-BY-4.0', 'uri': 'https://www.data.gv.at/info/netiquette?locale=de', 'description': 'Creative Commons Attribution 4.0 International'}], 'creators': [{'creator_n